In [7]:
import pandas as pd
import numpy as np
import re

print("Libraries Imported Successfully ✅")

Libraries Imported Successfully ✅


In [8]:
df = pd.read_csv("/Users/sarveshchoudhary/Documents/GitHub/fullstack_practice/Personal-Finance-Dashboard/data/raw/personal_transactions.csv")

print("First 5 Rows:")
display(df.head())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nShape of Dataset:")
print(df.shape)

First 5 Rows:


,Date,Description,Amount,Transaction Type,Category,Account Name
0,01/01/2018,Amazon,11.11,debit,Shopping,Platinum Card
1,01/02/2018,Mortgage Payment,1247.44,debit,Mortgage & Rent,Checking
2,01/02/2018,Thai Restaurant,24.22,debit,Restaurants,Silver Card
3,01/03/2018,Credit Card Payment,2298.09,credit,Credit Card Payment,Platinum Card
4,01/04/2018,Netflix,11.76,debit,Movies & DVDs,Platinum Card



Data Types:
Date                    str
Description             str
Amount              float64
Transaction Type        str
Category                str
Account Name            str
dtype: object

Missing Values:
Date                0
Description         0
Amount              0
Transaction Type    0
Category            0
Account Name        0
dtype: int64

Shape of Dataset:
(806, 6)


In [9]:
# --- Fix dates ---
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df.dropna(subset=['Date'], inplace=True)

# --- Fix amount column ---
df['Amount'] = df['Amount'].astype(str).str.replace(r'[₹$,]', '', regex=True)
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df.dropna(subset=['Amount'], inplace=True)

# --- Normalize debit/credit ---
df['Amount'] = df.apply(
    lambda r: -abs(r['Amount']) if str(r['Transaction Type']).lower() == 'debit'
    else abs(r['Amount']),
    axis=1
)

print("Data Cleaned Successfully ✅")
display(df.head())

Data Cleaned Successfully ✅


,Date,Description,Amount,Transaction Type,Category,Account Name
0,2018-01-01,Amazon,-11.11,debit,Shopping,Platinum Card
1,2018-02-01,Mortgage Payment,-1247.44,debit,Mortgage & Rent,Checking
2,2018-02-01,Thai Restaurant,-24.22,debit,Restaurants,Silver Card
3,2018-03-01,Credit Card Payment,2298.09,credit,Credit Card Payment,Platinum Card
4,2018-04-01,Netflix,-11.76,debit,Movies & DVDs,Platinum Card


In [10]:
df['month'] = df['Date'].dt.to_period('M').astype(str)
df['month_name'] = df['Date'].dt.strftime('%b %Y')
df['week'] = df['Date'].dt.isocalendar().week
df['year'] = df['Date'].dt.year
df['day_of_week'] = df['Date'].dt.day_name()
df['quarter'] = df['Date'].dt.to_period('Q').astype(str)

print("Time Columns Created ✅")
display(df.head())

Time Columns Created ✅


,Date,Description,Amount,Transaction Type,Category,Account Name,month,month_name,week,year,day_of_week,quarter
0,2018-01-01,Amazon,-11.11,debit,Shopping,Platinum Card,2018-01,Jan 2018,1,2018,Monday,2018Q1
1,2018-02-01,Mortgage Payment,-1247.44,debit,Mortgage & Rent,Checking,2018-02,Feb 2018,5,2018,Thursday,2018Q1
2,2018-02-01,Thai Restaurant,-24.22,debit,Restaurants,Silver Card,2018-02,Feb 2018,5,2018,Thursday,2018Q1
3,2018-03-01,Credit Card Payment,2298.09,credit,Credit Card Payment,Platinum Card,2018-03,Mar 2018,9,2018,Thursday,2018Q1
4,2018-04-01,Netflix,-11.76,debit,Movies & DVDs,Platinum Card,2018-04,Apr 2018,13,2018,Sunday,2018Q2


In [11]:
df['transaction_type'] = df['Amount'].apply(
    lambda x: 'Income' if x > 0 else 'Expense'
)

print("Income vs Expense Created ✅")
display(df.head())

Income vs Expense Created ✅


,Date,Description,Amount,Transaction Type,Category,Account Name,month,month_name,week,year,day_of_week,quarter,transaction_type
0,2018-01-01,Amazon,-11.11,debit,Shopping,Platinum Card,2018-01,Jan 2018,1,2018,Monday,2018Q1,Expense
1,2018-02-01,Mortgage Payment,-1247.44,debit,Mortgage & Rent,Checking,2018-02,Feb 2018,5,2018,Thursday,2018Q1,Expense
2,2018-02-01,Thai Restaurant,-24.22,debit,Restaurants,Silver Card,2018-02,Feb 2018,5,2018,Thursday,2018Q1,Expense
3,2018-03-01,Credit Card Payment,2298.09,credit,Credit Card Payment,Platinum Card,2018-03,Mar 2018,9,2018,Thursday,2018Q1,Income
4,2018-04-01,Netflix,-11.76,debit,Movies & DVDs,Platinum Card,2018-04,Apr 2018,13,2018,Sunday,2018Q2,Expense


In [12]:
CATEGORY_RULES = {
    'Food & Dining': ['swiggy', 'zomato', 'restaurant', 'cafe', 'pizza', 'burger'],
    'Groceries': ['bigbasket', 'blinkit', 'dmart', 'supermarket'],
    'Transport': ['uber', 'ola', 'metro', 'petrol', 'fuel'],
    'Shopping': ['amazon', 'flipkart', 'myntra', 'ajio'],
    'Entertainment': ['netflix', 'spotify', 'hotstar', 'prime video'],
    'Utilities': ['electricity', 'jio', 'airtel', 'water bill'],
    'Health': ['pharmacy', 'hospital', 'clinic', 'apollo'],
    'Education': ['udemy', 'coursera', 'books', 'college fee'],
    'Finance & Banking': ['emi', 'loan', 'insurance', 'sip'],
    'Rent & Housing': ['rent', 'maintenance', 'pg'],
    'Income': ['salary', 'refund', 'cashback', 'interest credit']
}

def categorize(description):
    desc = str(description).lower()
    for category, keywords in CATEGORY_RULES.items():
        if any(re.search(kw, desc) for kw in keywords):
            return category
    return 'Other'

In [13]:
df['category'] = df['Description'].apply(categorize)

uncategorized = df[df['category'] == 'Other']

print(f"Uncategorized Rows: {len(uncategorized)}")

display(uncategorized[['Description']].head(20))

print("Categories Applied Successfully ✅")

Uncategorized Rows: 255


,Description
1,Mortgage Payment
3,Credit Card Payment
5,American Tavern
6,Hardware Store
7,Gas Company
8,Hardware Store
10,Phone Company
11,Shell
12,Grocery Store
13,Biweekly Paycheck


Categories Applied Successfully ✅


In [14]:
expenses = df[df['transaction_type'] == 'Expense'].copy()
income = df[df['transaction_type'] == 'Income'].copy()

monthly_income = income.groupby('month')['Amount'].sum().rename('total_income')

monthly_expense = expenses.groupby('month')['Amount'].apply(
    lambda x: abs(x.sum())
).rename('total_expense')

monthly_kpi = pd.concat([monthly_income, monthly_expense], axis=1).fillna(0)

monthly_kpi['savings'] = monthly_kpi['total_income'] - monthly_kpi['total_expense']

monthly_kpi['savings_rate'] = (
    monthly_kpi['savings'] / monthly_kpi['total_income'] * 100
).round(2)

monthly_kpi.reset_index(inplace=True)

display(monthly_kpi.tail())

print("Monthly KPI Created ✅")

,month,total_income,total_expense,savings,savings_rate
19,2019-11,360.56,225.96,134.60,37.33
20,2019-12,2000.00,767.06,1232.94,61.65
21,2018-01,0.00,606.21,-606.21,-inf
22,2018-10,0.00,743.53,-743.53,-inf
23,2019-08,0.00,772.10,-772.10,-inf


Monthly KPI Created ✅


In [15]:
category_summary = (
    expenses.groupby(['month', 'category'])['Amount']
    .apply(lambda x: abs(x.sum()))
    .reset_index()
    .rename(columns={'Amount': 'spent'})
)

display(category_summary.head())

print("Category Summary Created ✅")

,month,category,spent
0,2018-01,Food & Dining,8.00
1,2018-01,Other,444.69
2,2018-01,Shopping,153.52
3,2018-02,Food & Dining,24.22
4,2018-02,Other,12552.25


Category Summary Created ✅


In [16]:
df.to_csv("transactions_clean.csv", index=False)
monthly_kpi.to_csv("monthly_kpi.csv", index=False)
category_summary.to_csv("category_summary.csv", index=False)

print("All Files Exported Successfully ✅")

All Files Exported Successfully ✅


In [17]:
from google.colab import files

files.download("transactions_clean.csv")
files.download("monthly_kpi.csv")
files.download("category_summary.csv")

ModuleNotFoundError: No module named 'google.colab'